![NWM](img/NWM.png)

# Watershed-Scale Analysis of National Water Model Snow Data
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org)
Last updated:Oct 16, 2025

**Introduction**: This notebook demonstrates how to perform a watershed-scale comparison between modeled and observed SWE at individual SNOTEL sites. It uses National Water Model and Snotel data that is collected in the `01_data_collection.ipynb` notebook. 

### 1. Prepare the Python Environment

### 2. Subset the modeled dataset to the watershed of interest

In this notebook, we take a deeper dive into analyzing the NWM snow data across the entire Tuolumne River watershed. To do so, we'll first need to clip the gridded dataset to the watershed boundary, enabling us to compute spatial averages over the region. To begin, we will create a single geometry for the watershed by dissolving all HUC8-level basins within it.

In [1]:
# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2020-09-30'

# Path to the watershed shapefile
watershed = "./domain_data/TolumneRiver_18040009.shp"

# Extract the bounding box coordinates of a watershed
watershed_gdf = gpd.read_file(os.path.join(os.getcwd(), watershed)).to_crs(epsg=4326)

# Create a single geometry for the watershed by dissolving all HUC8-level basins within it.
watershed_huc8 = watershed_gdf.dissolve(by='HUC_8')

We first ensure that the NWM data is loaded and that the domain of interest uses the same coordinate reference system (CRS) as the NWM dataset. This is required when clipping the dataset using rioxarray’s clip method later 

In [ ]:
%time 
ds = xr.open_zarr(store=conus_bucket_url, 
                        consolidated=True, 
                        storage_options={
                        "anon": True,
                        "client_kwargs": {"region_name": "us-east-1"}
                    })
ds.rio.set_crs(ds.crs.attrs['esri_pe_string'])
ds.rio.write_crs(inplace=True);
watershed_huc8_prj = watershed_huc8.to_crs(ds.rio.crs)

Let's clip the gridded data to the extent of this watershed. This can be done using `rioxarray`'s "clip" method.

In [ ]:
%%time
ds_clip = ds.rio.clip(
         watershed_huc8_prj.geometry.values,
         watershed_huc8_prj.crs,
         all_touched=True,   # select all grid cells that touch the vector boundary
         drop=True,          # drop anything that is outside the clipped region
         invert=False,
         from_disk=True).sel(time=slice(StartDate, EndDate))

Preview the data at a specific point in time by defining an **exact date and time** (e.g., '2019-02-01T00:00') or an **index** (e.g., 1000, referring to the 1000th time step in the data array). We will use `hvplot` and `Holoviews` Python packages to create a more informative plot that displays both the gridded data and the vector data. One important note: for `hvplot` to visualize gridded data correctly on a map, all data must be in a **geographic coordinate system** (e.g., EPSG:4326). Note that it may take a while for the following task (map visualization) to be completed.

In [ ]:
# Explore the time values
ds_clip.time.values

In [ ]:
# Explore the variable of interest. We will be working with SNEQV, which is SWE. But feel free to explore other varibales. Note that for some of these varibales, the dimension is different and the funciton may not work for all.
ds_clip.data_vars

In [ ]:
# Define the time and variable of interest
data_var='SNEQV'
time_index='2019-01-01T00:00:00.000000000'

nwm_utils.plot_grid_vector_data(ds_clip, data_var, time_index, watershed_huc8, gdf_in_bbox)

Go ahead and play around with the time slider above to see how SWE changes over time across the watershed. 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🔍 Explore</h4>
  <p>
    Using the zoom options in the hvplot toolbar, can you comment on the possible reasons for the missing values or pixels visible on the map?
  </p>
</div>

### 3. Compare the spatial average SWE across the entire watershed

This analysis requires us to compute the spatial average for both observed and modeled snow water equivilent data and create a single Pandas DataFrame that combines the basin-averaged modeled data with the observed data for direct comparison. To do this, we'll read the observed data collected in the `01_data_collection.ipynb` notebook. 

In [ ]:
# Observed data
df_obs_swe_mean = nwm_utils.compute_spatial_agg_from_obs('./obs_outputs', 'mean')
df_obs_swe_mean['Date'] = pd.to_datetime(df_obs_swe_mean['Date']).dt.date
df_mod_swe_mean['Date_Local'] = pd.to_datetime(df_mod_swe_mean['Date_Local']).dt.date

In [ ]:
# Modeled data
ds_clip_swe_mean = ds_clip.SNEQV.mean(dim=(['x', 'y'])).sel(time=slice(StartDate, EndDate)).compute()
df_mod_swe_mean = nwm_utils.prep_nwm_swe_dataframe(ds_clip_swe_mean, gdf_in_bbox.iloc[0].state)

In [ ]:
# Combine
combined_df_mean = pd.merge(
    df_mod_swe_mean, df_obs_swe_mean, 
    left_on='Date_Local',  
    right_on='Date',       
    how='inner'  
)
combined_df_mean = combined_df_mean[['Date_Local', 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters']]
combined_df_mean.index = pd.to_datetime(combined_df_mean['Date_Local'])


Create plots and compute stats

In [ ]:
# Plot
nwm_utils.comparison_plots(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

In [ ]:
# Stats
nwm_utils.compute_stats(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What could be the potential sources of discrepancies between the spatially averaged SWE from the modeled and observed datasets within this catchment?  
How could you generate more reliable or informative statistics?  
How do you think the presence of SWE values equal to zero across large areas of the gridded dataset (as seen in the map above) — particularly when this pattern is consistent across the water year — might impact the spatially averaged SWE values?  
How could this influence your interpretation?
  </p>
</div>

### 4. Compare the spatial average SWE across a catchment within the watershed

#### Don Pedro Reservoir Basin

Repeat the steps above, but this time focus on a smaller domain. For example, the upstream catchments of the Don Pedro Reservoir (DonPedroDam_Upstream_Basin.shp). This allows us to examine the impact of excluding areas where SWE is zero for the majority of days within a water year. Could this approach improve the statistical comparison between modeled and observed SWE?

In [ ]:
donpedro_gdf = gpd.read_file(os.path.join(os.getcwd(), basin)).to_crs(epsg=4326)
donpedro_gdf_prj = donpedro_gdf.to_crs(ds.rio.crs)
ds_clip_donpedro = ds.rio.clip(
         donpedro_gdf_prj.geometry.values,
         donpedro_gdf_prj.crs,
         all_touched=True,   
         drop=True,         
         invert=False,
         from_disk=True).sel(time=slice(StartDate, EndDate))

In [ ]:
# Define the time and variable of interest
data_var='SNEQV'
time_index='2019-01-01T00:00:00.000000000'

nwm_utils.plot_grid_vector_data(ds_clip_donpedro, data_var, time_index, donpedro_gdf, gdf_in_bbox)

In [ ]:
# Modeled data
ds_clip_swe_mean = ds_clip_donpedro.SNEQV.mean(dim=(['x', 'y'])).sel(time=slice(StartDate, EndDate)).compute()
df_mod_swe_mean = nwm_utils.prep_nwm_swe_dataframe(ds_clip_swe_mean, gdf_in_bbox.iloc[0].state)

# Observed data
df_obs_swe_mean = nwm_utils.compute_spatial_agg_from_obs('./obs_outputs', 'mean')
df_obs_swe_mean['Date'] = pd.to_datetime(df_obs_swe_mean['Date']).dt.date
df_mod_swe_mean['Date_Local'] = pd.to_datetime(df_mod_swe_mean['Date_Local']).dt.date

# Combine with the observed data
combined_df_mean = pd.merge(
    df_mod_swe_mean, df_obs_swe_mean, 
    left_on='Date_Local',  
    right_on='Date',       
    how='inner'  
)
combined_df_mean = combined_df_mean[['Date_Local', 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters']]
combined_df_mean.index = pd.to_datetime(combined_df_mean['Date_Local'])

# Plot
nwm_utils.comparison_plots(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

In [ ]:
# Stats
nwm_utils.compute_stats(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

Visualize both the temporal and spatial variability of modeled SWE across the domain. To achieve this, we first create a subset of the modeled data by selecting only the first time step of each month from the simulated SWE dataset. To improve processing efficiency, we chunk the dataset along the time dimension, allowing faster access to individual time slices. We then apply a time filter and compute each selected slice individually to avoid excessive memory use. Finally, we combine the computed slices back into a single dataset for analysis and visualization.

In [ ]:
# Check the current chunks
ds_clip_donpedro.chunks

In [ ]:
# Rechunck
ds_clip_donpedro = ds_clip_donpedro.chunk({'time': 1})

**Note**: The following code may take several minutes to run.

In [ ]:
%%time 

# Apply time filter
mask = (ds_clip_donpedro.time.dt.is_month_start) & (ds_clip_donpedro.time.dt.hour == 0)

# Compute in smaller pieces
selected_times = ds_clip_donpedro.time[mask]
parts = []
for t in selected_times.values:
    part = ds_clip_donpedro.sel(time=t).compute()  # One slice at a time
    parts.append(part)

# Combine the pieces
ds_monthly_start = xr.concat(parts, dim='time')

In [ ]:
data_var='SNEQV'
nwm_utils.plot_grid_vector_monthly_data(ds_monthly_start, data_var, donpedro_gdf, gdf_in_bbox)

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    In the plot above, you can see that in both accumulation and melt periods (e.g., Feb-Jun), snow was present in areas not covered by the existing observation points.  
How would having more observation points distributed across these snow-covered areas affect the basin-average SWE from observations, as calculated and plotted previously (`combined_df_mean`)?  
How could this change influence the bias, Nash-Sutcliffe Efficiency (NSE), and other statistical comparisons between modeled and observed SWE?  
And, how might improving the capture of spatial variability in SWE affect forecasts under future climate conditions?
  </p>
</div>